In [1]:
"""
itr_ranking.py
==============
Baixa os dados ITR da CVM para um ano inteiro, calcula EV/EBIT e ROIC
de TODAS as empresas disponíveis e retorna um ranking por trimestre.

Uso:
    from itr_ranking import processar_ano

    df_2023 = processar_ano(2023)
    df_2023.to_csv("ranking_2023.csv", index=False, encoding="utf-8-sig")

Nota sobre EV:
    EV = Market Cap + Dívida Líquida.
    Market Cap exige preço de mercado, que não está nos arquivos CVM.
    Por isso retornamos também o ranking por ROIC puro (sem preço)
    e deixamos o EV/EBIT como coluna opcional a ser preenchida depois.

Instalação:
    pip install pandas requests
"""

import requests
import numpy as np
import pandas as pd
from io import BytesIO
from zipfile import ZipFile

BASE_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"

# Contas CVM
CONTAS_DRE = {"ebit": "3.05", "ll": "3.11", "ir_csll": "3.08"}
CONTAS_BPA = {"caixa": "1.01.01", "aplic_cp": "1.01.02"}
CONTAS_BPP = {"divida_cp": "2.01.04", "divida_lp": "2.02.01", "pl": "2.03"}

TODAS_CONTAS = {**CONTAS_DRE, **CONTAS_BPA, **CONTAS_BPP}

# ---------------------------------------------------------------------------
# 1. Download do ZIP anual e extração dos CSVs internos
# ---------------------------------------------------------------------------

def baixar_zip_ano(ano: int) -> dict[str, pd.DataFrame] | None:
    """
    Baixa itr_cia_aberta_{ano}.zip e extrai os três CSVs relevantes:
        itr_cia_aberta_DRE_con_{ano}.csv
        itr_cia_aberta_BPA_con_{ano}.csv
        itr_cia_aberta_BPP_con_{ano}.csv

    Retorna dicionário {'dre': df, 'bpa': df, 'bpp': df} ou None se falhar.
    """
    url = f"{BASE_URL}/itr_cia_aberta_{ano}.zip"
    print(f"Baixando {url} ...")

    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"  ✗ Erro no download: {e}")
        return None

    alvos = {
        "dre": f"itr_cia_aberta_DRE_con_{ano}.csv",
        "bpa": f"itr_cia_aberta_BPA_con_{ano}.csv",
        "bpp": f"itr_cia_aberta_BPP_con_{ano}.csv",
    }

    resultado = {}
    with ZipFile(BytesIO(r.content)) as z:
        arquivos_zip = z.namelist()
        print(f"  Arquivos no ZIP: {arquivos_zip}")

        for chave, nome_csv in alvos.items():
            if nome_csv not in arquivos_zip:
                print(f"  ✗ {nome_csv} não encontrado no ZIP")
                continue
            df = pd.read_csv(
                z.open(nome_csv), sep=";", encoding="latin1", dtype=str
            )
            resultado[chave] = df
            print(f"  ✓ {nome_csv}: {len(df):,} linhas")

    return resultado if resultado else None


# ---------------------------------------------------------------------------
# 2. Limpeza
# ---------------------------------------------------------------------------

def _limpar(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Mantém apenas DF Consolidado (quando disponível) ou Individual
    - Remove período anterior (PENÚLTIMO)
    - Desduplicata versões do mesmo ITR (fica a mais recente)
    - Converte tipos e normaliza escala para R$
    """
    df = df.copy()

    # Mantém só ÚLTIMO exercício
    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]

    # Prefere consolidado; onde não existe, aceita individual
    tem_consolidado = df[
        df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    ]["CNPJ_CIA"].unique()

    mask_cons = df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    mask_ind  = ~df["CNPJ_CIA"].isin(tem_consolidado)
    df = df[mask_cons | mask_ind]

    # Desduplicata versões — fica a mais recente
    df["VERSAO"] = pd.to_numeric(df["VERSAO"], errors="coerce")
    df = (
        df.sort_values("VERSAO", ascending=False)
          .drop_duplicates(
              subset=["CNPJ_CIA", "DT_FIM_EXERC", "CD_CONTA"],
              keep="first",
          )
    )

    # Tipos
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    df["VL_CONTA"]     = pd.to_numeric(df["VL_CONTA"], errors="coerce")

    # Normaliza escala → R$
    mask_mil = df["ESCALA_MOEDA"].str.upper().str.contains("MIL", na=False)
    df.loc[mask_mil, "VL_CONTA"] *= 1000

    return df


# ---------------------------------------------------------------------------
# 3. Pivotamento: cada CD_CONTA vira uma coluna
# ---------------------------------------------------------------------------

def _pivotar(df: pd.DataFrame, contas: dict) -> pd.DataFrame:
    """
    Filtra as contas desejadas e pivota para colunas nomeadas.
    Retorna [CNPJ_CIA, DENOM_CIA, DT_FIM_EXERC, col1, col2, ...]
    """
    mapa_inv = {v: k for k, v in contas.items()}  # CD_CONTA → nome legível

    sub = df[df["CD_CONTA"].isin(contas.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()

    piv.columns.name = None
    return piv


# ---------------------------------------------------------------------------
# 4. Cálculo dos indicadores
# ---------------------------------------------------------------------------

def _calcular(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula ROIC e as peças necessárias para EV/EBIT.
    EV/EBIT fica como NaN até que preços sejam injetados externamente.
    """
    df = df.copy()

    # --- Componentes de balanço ---
    dispon     = df.get("caixa",    pd.Series(0, index=df.index)).fillna(0) \
                + df.get("aplic_cp", pd.Series(0, index=df.index)).fillna(0)
    div_bruta  = df.get("divida_cp", pd.Series(0, index=df.index)).fillna(0) \
                + df.get("divida_lp", pd.Series(0, index=df.index)).fillna(0)
    div_liq    = div_bruta - dispon
    pl         = df.get("pl", pd.Series(np.nan, index=df.index))
    ebit       = df.get("ebit", pd.Series(np.nan, index=df.index))
    ll         = df.get("ll",   pd.Series(np.nan, index=df.index))
    ir         = df.get("ir_csll", pd.Series(0, index=df.index)).fillna(0).abs()

    df["disponibilidades"] = dispon
    df["divida_bruta"]     = div_bruta
    df["divida_liquida"]   = div_liq

    # --- Alíquota efetiva ---
    base_ir = ll.abs() + ir
    aliq = np.where(base_ir > 0, ir / base_ir, 0.34)
    df["aliquota_efetiva"] = np.clip(aliq, 0, 0.50)

    # --- NOPAT e Capital Investido ---
    df["nopat"]             = ebit * (1 - df["aliquota_efetiva"])
    cap_inv                 = pl.fillna(0) + div_liq
    df["capital_investido"] = cap_inv

    # --- ROIC ---
    df["roic"] = np.where(cap_inv.abs() > 1e-6, df["nopat"] / cap_inv, np.nan)

    # --- EV/EBIT: market_cap virá de fora; por ora NaN ---
    df["market_cap"] = np.nan   # preencher externamente se desejar
    df["ev"]         = np.nan   # market_cap + divida_liquida
    df["ev_ebit"]    = np.nan   # ev / ebit

    return df


# ---------------------------------------------------------------------------
# 5. Ranking por trimestre
# ---------------------------------------------------------------------------

def _rankear(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada trimestre (DT_FIM_EXERC), cria rankings de ROIC.
    Menor rank = melhor ROIC.
    Empresas com ROIC negativo ou nulo ficam no fim do ranking.
    """
    df = df.copy()

    df["rank_roic"] = (
        df.groupby("DT_FIM_EXERC")["roic"]
          .rank(ascending=False, method="min", na_option="bottom")
          .astype("Int64")
    )

    # Quando EV/EBIT estiver disponível, descomentar:
    # df["rank_ev_ebit"] = (
    #     df.groupby("DT_FIM_EXERC")["ev_ebit"]
    #       .rank(ascending=True, method="min", na_option="bottom")
    #       .astype("Int64")
    # )
    # df["rank_magic"] = df["rank_roic"] + df["rank_ev_ebit"]

    return df.sort_values(["DT_FIM_EXERC", "rank_roic"])


# ---------------------------------------------------------------------------
# 6. Pipeline principal — um ano de cada vez
# ---------------------------------------------------------------------------

def processar_ano(ano: int) -> pd.DataFrame:
    """
    Pipeline completo para um ano.

    Retorna DataFrame com colunas:
        CNPJ_CIA, DENOM_CIA, DT_FIM_EXERC,
        ebit, nopat, disponibilidades, divida_bruta, divida_liquida,
        pl, capital_investido, roic, rank_roic,
        market_cap(*), ev(*), ev_ebit(*)   ← (*) NaN até injetar preços

    Exemplo de uso para vários anos:
        frames = [processar_ano(a) for a in range(2019, 2025)]
        historico = pd.concat(frames, ignore_index=True)
    """
    # Download
    dados = baixar_zip_ano(ano)
    if not dados:
        return pd.DataFrame()

    # Limpeza
    print("Limpando dados...")
    dre = _limpar(dados["dre"])
    bpa = _limpar(dados["bpa"])
    bpp = _limpar(dados["bpp"])

    # Pivotamento
    print("Pivotando contas...")
    df_dre = _pivotar(dre, CONTAS_DRE)
    df_bpa = _pivotar(bpa, CONTAS_BPA)
    df_bpp = _pivotar(bpp, CONTAS_BPP)

    # Merge dos três demonstrativos
    chave = ["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"]
    base = (
        df_dre
        .merge(df_bpa, on=chave, how="outer")
        .merge(df_bpp, on=chave, how="outer")
    )
    print(f"  {base['CNPJ_CIA'].nunique():,} empresas | "
          f"{base['DT_FIM_EXERC'].nunique()} trimestres")

    # Cálculo e ranking
    print("Calculando indicadores...")
    resultado = _calcular(base)
    resultado = _rankear(resultado)

    # Colunas finais
    cols = [
        "CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC",
        "ebit", "nopat",
        "disponibilidades", "divida_bruta", "divida_liquida",
        "pl", "capital_investido",
        "roic", "rank_roic",
        "market_cap", "ev", "ev_ebit",
    ]
    cols_presentes = [c for c in cols if c in resultado.columns]
    resultado = resultado[cols_presentes]

    print(f"✅ Ano {ano} concluído: {len(resultado):,} linhas\n")
    return resultado

In [2]:
"""
itr_ranking.py
==============
Baixa ITR + DFP da CVM, calcula EBIT/EBITDA/ROIC anualizados
de TODAS as empresas não-financeiras e retorna ranking por trimestre.

Correções aplicadas:
    1. DFP incluído → cobre Q4 (dezembro) para todas as empresas
    2. EBIT anualizado via DT_INI_EXERC → elimina distorção entre trimestres
    3. Setor financeiro filtrado → bancos/seguradoras excluídos
    4. Receita (3.01) e D&A (DFC) extraídas → margem líquida e EBITDA
    5. QT_ACAO_TOTAL_CAP_INTEGR extraída do índice CVM → LPA/VPA/Graham e market_cap
    6. SETOR_ATIV anexado → permite ranking por setor e isenção de infra
"""

import requests
import numpy as np
import pandas as pd
from io import BytesIO
from zipfile import ZipFile

# ---------------------------------------------------------------------------
# Contas CVM
# ---------------------------------------------------------------------------
CONTAS_DRE = {"receita": "3.01", "ebit": "3.05", "ir_csll": "3.08", "ll": "3.11",
              "ll_alt": "3.09"}  # 3.09 = Lucro Líquido em bancos (DRE financeira)
CONTAS_BPA = {"caixa": "1.01.01", "aplic_cp": "1.01.02"}
CONTAS_BPP = {"divida_cp": "2.01.04", "divida_lp": "2.02.01", "pl": "2.03",
              "pl_fin": "2.08"}  # 2.08 = Patrimônio Líquido em bancos (BPP financeiro)

ITR_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"
DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS"
CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"

# Setores financeiros a excluir (campo SETOR_ATIV do cadastro CVM)
SETORES_FINANCEIROS = {
    "Bancos",
    "Intermediários Financeiros",
    "Previdência e Seguros",
    "Auxiliares Financeiros",
    "Outros Intermediários Financeiros e Serviços Relacionados",
}

# ---------------------------------------------------------------------------
# 1. Download — ITR + DFP
# ---------------------------------------------------------------------------

def _ler_csv_do_zip(content: bytes, nome_csv: str) -> pd.DataFrame | None:
    try:
        with ZipFile(BytesIO(content)) as z:
            if nome_csv not in z.namelist():
                return None
            return pd.read_csv(
                z.open(nome_csv), sep=";", encoding="latin1", dtype=str
            )
    except Exception as e:
        print(f"    ✗ Erro ao ler {nome_csv}: {e}")
        return None


def _extrair_qt_acoes(content: bytes, prefixo: str, ano: int) -> pd.DataFrame:
    """
    Lê {prefixo}_cia_aberta_composicao_capital_{ano}.csv dentro do ZIP e extrai
    QT_ACAO_TOTAL_CAP_INTEGR (total de ações) por CNPJ_CIA + DT_REFER (→ DT_FIM_EXERC).

    Defensivo: se a coluna não existir nessa fonte, avisa e devolve DataFrame
    vazio (o pipeline continua, qt_acoes fica NaN).
    """
    nome_csv = f"{prefixo}_cia_aberta_composicao_capital_{ano}.csv"
    df = _ler_csv_do_zip(content, nome_csv)
    if df is None:
        print(f"    ⚠ {nome_csv} não encontrado — qt_acoes ficará NaN")
        return pd.DataFrame(columns=["CNPJ_CIA", "DT_FIM_EXERC", "qt_acoes"])

    col = "QT_ACAO_TOTAL_CAP_INTEGR"
    if col not in df.columns or "DT_REFER" not in df.columns:
        print(f"    ⚠ {col}/DT_REFER ausente em {nome_csv} — qt_acoes ficará NaN")
        return pd.DataFrame(columns=["CNPJ_CIA", "DT_FIM_EXERC", "qt_acoes"])

    df = df.copy()
    df["VERSAO"]       = pd.to_numeric(df.get("VERSAO"), errors="coerce")
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_REFER"], errors="coerce")
    # remove eventuais separadores (ações são inteiras) antes de converter
    df["qt_acoes"]     = pd.to_numeric(
        df[col].astype(str).str.replace(r"\D", "", regex=True).replace("", np.nan),
        errors="coerce",
    )

    df = (
        df.dropna(subset=["DT_FIM_EXERC"])
          .sort_values("VERSAO", ascending=False)
          .drop_duplicates(subset=["CNPJ_CIA", "DT_FIM_EXERC"], keep="first")
    )
    return df[["CNPJ_CIA", "DT_FIM_EXERC", "qt_acoes"]]


def baixar_dados(ano: int) -> dict[str, pd.DataFrame]:
    """
    Baixa ITR (Q1–Q3) e DFP (Q4) do ano e retorna {'dre','bpa','bpp','dfc','qt_acoes'}.
    ITR cobre março/junho/setembro; DFP cobre dezembro (ano fiscal padrão).
    """
    frames: dict[str, list] = {"dre": [], "bpa": [], "bpp": [], "dfc": [], "qt_acoes": []}
    tipos  = {"dre": "DRE_con", "bpa": "BPA_con", "bpp": "BPP_con", "dfc": "DFC_MI_con"}

    for fonte, base_url in [("ITR", ITR_URL), ("DFP", DFP_URL)]:
        prefixo = "itr" if fonte == "ITR" else "dfp"
        zip_url = f"{base_url}/{prefixo}_cia_aberta_{ano}.zip"
        print(f"  Baixando {fonte} {ano}...", end=" ", flush=True)
        try:
            r = requests.get(zip_url, timeout=120)
            r.raise_for_status()
            print("✓")
        except Exception as e:
            print(f"✗ ({e})")
            continue

        for chave, sufixo in tipos.items():
            nome_csv = f"{prefixo}_cia_aberta_{sufixo}_{ano}.csv"
            df = _ler_csv_do_zip(r.content, nome_csv)
            if df is not None:
                df["_fonte"] = fonte   # marca origem para deduplicação posterior
                frames[chave].append(df)

        # Índice principal → total de ações
        qt = _extrair_qt_acoes(r.content, prefixo, ano)
        if not qt.empty:
            frames["qt_acoes"].append(qt)

    return {
        chave: pd.concat(dfs, ignore_index=True)
        for chave, dfs in frames.items()
        if dfs
    }


# ---------------------------------------------------------------------------
# 2. Cadastro CVM → SETOR_ATIV + CNPJs financeiros
# ---------------------------------------------------------------------------

def _carregar_cadastro() -> pd.DataFrame:
    """Retorna [CNPJ_CIA, SETOR_ATIV] do cadastro CVM (1 download)."""
    try:
        cad = pd.read_csv(CAD_URL, sep=";", encoding="latin1", dtype=str)
        cad.columns = cad.columns.str.strip()
        cad["CNPJ_CIA"]   = cad["CNPJ_CIA"].str.strip()
        cad["SETOR_ATIV"] = cad["SETOR_ATIV"].str.strip()
        cad = cad.dropna(subset=["CNPJ_CIA"]).drop_duplicates("CNPJ_CIA")
        return cad[["CNPJ_CIA", "SETOR_ATIV"]]
    except Exception as e:
        print(f"  ⚠ Não foi possível carregar o cadastro: {e}")
        return pd.DataFrame(columns=["CNPJ_CIA", "SETOR_ATIV"])


# ---------------------------------------------------------------------------
# 3. Limpeza
# ---------------------------------------------------------------------------

def _limpar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]

    df["DT_FIM_EXERC_dt"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    mask_itr_dez = (df["_fonte"] == "ITR") & (df["DT_FIM_EXERC_dt"].dt.month == 12)
    df = df[~mask_itr_dez].drop(columns=["DT_FIM_EXERC_dt"])

    tem_cons = df[df["GRUPO_DFP"].str.contains("Consolidado", na=False)]["CNPJ_CIA"].unique()
    mask_cons = df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    mask_ind  = ~df["CNPJ_CIA"].isin(tem_cons)
    df = df[mask_cons | mask_ind]

    df["VERSAO"]       = pd.to_numeric(df["VERSAO"], errors="coerce")
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    df["VL_CONTA"]     = pd.to_numeric(df["VL_CONTA"], errors="coerce")

    # DT_INI_EXERC só existe na DRE/DFC (fluxos), não no balanço
    tem_ini = "DT_INI_EXERC" in df.columns
    if tem_ini:
        df["DT_INI_EXERC"] = pd.to_datetime(df["DT_INI_EXERC"], errors="coerce")

    # Desduplicação: maior VERSAO; entre versões iguais mantém o período mais
    # longo (DT_INI mais antigo = acumulado YTD), evitando misturar o trimestre
    # isolado (3 meses) com o acumulado na mesma data — fonte de linhas
    # duplicadas com dados parciais após o merge.
    sort_cols, asc = ["VERSAO"], [False]
    if tem_ini:
        sort_cols.append("DT_INI_EXERC"); asc.append(True)
    df = (
        df.sort_values(sort_cols, ascending=asc)
          .drop_duplicates(subset=["CNPJ_CIA", "DT_FIM_EXERC", "CD_CONTA"], keep="first")
    )

    mask_mil = df["ESCALA_MOEDA"].str.upper().str.contains("MIL", na=False)
    df.loc[mask_mil, "VL_CONTA"] *= 1000

    return df.drop(columns=["_fonte"], errors="ignore")


# ---------------------------------------------------------------------------
# 4. Pivotamento
# ---------------------------------------------------------------------------

def _pivotar_dre(df: pd.DataFrame) -> pd.DataFrame:
    """
    DRE: inclui DT_INI_EXERC no índice para calcular período real depois.
    """
    mapa_inv = {v: k for k, v in CONTAS_DRE.items()}
    sub = df[df["CD_CONTA"].isin(CONTAS_DRE.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DT_FIM_EXERC", "DT_INI_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()
    piv.columns.name = None
    return piv


def _pivotar_balanco(df: pd.DataFrame, contas: dict) -> pd.DataFrame:
    """Balanço: não precisa de DT_INI_EXERC (valores pontuais)."""
    mapa_inv = {v: k for k, v in contas.items()}
    sub = df[df["CD_CONTA"].isin(contas.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DT_FIM_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()
    piv.columns.name = None
    return piv


def _pivotar_da(df: pd.DataFrame) -> pd.DataFrame:
    """
    Depreciação & Amortização do DFC: identificadas por DS_CONTA (não há CD_CONTA
    padronizado). Soma todas as linhas cujo nome contém 'deprecia' ou 'amortiza'.
    DFC é YTD ⇒ mantém DT_INI_EXERC para anualizar junto com a DRE.
    """
    if "DS_CONTA" not in df.columns:
        return pd.DataFrame(columns=["CNPJ_CIA", "DT_FIM_EXERC", "DT_INI_EXERC", "d_a"])

    mask = df["DS_CONTA"].str.lower().str.contains("deprecia|amortiza", na=False, regex=True)
    sub = df[mask].copy()
    if sub.empty:
        return pd.DataFrame(columns=["CNPJ_CIA", "DT_FIM_EXERC", "DT_INI_EXERC", "d_a"])

    da = (
        sub.groupby(["CNPJ_CIA", "DT_FIM_EXERC", "DT_INI_EXERC"], as_index=False)["VL_CONTA"]
           .sum()
           .rename(columns={"VL_CONTA": "d_a"})
    )
    da["d_a"] = da["d_a"].abs()   # D&A no DFC costuma vir positiva (adicionada de volta)
    return da


# ---------------------------------------------------------------------------
# 5. Cálculo com anualização (EBIT, EBITDA, Receita, LL)
# ---------------------------------------------------------------------------

def _calcular(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Anualização: usa DT_INI_EXERC quando disponível ---
    # Após o merge outer, linhas sem correspondência na DRE terão DT_INI_EXERC NaN
    # Nesses casos assume 12 meses (fator 1.0 = sem alteração)
    if "DT_INI_EXERC" in df.columns:
        meses = (
            (df["DT_FIM_EXERC"] - df["DT_INI_EXERC"])
            / pd.Timedelta(days=30.4375)
        ).clip(lower=1).round()
        meses = meses.fillna(12)   # ← fallback para linhas sem DT_INI_EXERC
    else:
        meses = pd.Series(12, index=df.index)

    fator = (12 / meses).clip(upper=4)
    df["meses_periodo"] = meses

    # --- Itens de fluxo (DRE/DFC) → anualizados ---
    ebit_raw    = df.get("ebit",    pd.Series(np.nan, index=df.index))
    ll          = df.get("ll",      pd.Series(np.nan, index=df.index))
    # Bancos não usam a conta 3.11; o lucro líquido vem da 3.09 → fallback
    ll          = ll.where(ll.notna(), df.get("ll_alt", pd.Series(np.nan, index=df.index)))
    receita     = df.get("receita", pd.Series(np.nan, index=df.index))
    d_a         = df.get("d_a",     pd.Series(0,      index=df.index)).fillna(0)
    ir          = df.get("ir_csll", pd.Series(0,      index=df.index)).fillna(0).abs()

    df["ebit_anualizado"]    = ebit_raw * fator
    df["ll_anualizado"]      = ll * fator
    df["receita_anualizada"] = receita * fator
    df["d_a_anualizado"]     = d_a * fator
    df["ebitda"]             = df["ebit_anualizado"] + df["d_a_anualizado"]

    ebit = df["ebit_anualizado"]

    # --- Itens de balanço (pontuais) → NÃO anualizam ---
    dispon    = df.get("caixa",     pd.Series(0, index=df.index)).fillna(0) \
              + df.get("aplic_cp",  pd.Series(0, index=df.index)).fillna(0)
    div_bruta = df.get("divida_cp", pd.Series(0, index=df.index)).fillna(0) \
              + df.get("divida_lp", pd.Series(0, index=df.index)).fillna(0)
    div_liq   = div_bruta - dispon
    pl_nonfin = df.get("pl",     pd.Series(np.nan, index=df.index))
    pl_fin    = df.get("pl_fin", pd.Series(np.nan, index=df.index))
    # Bancos: PL está na conta 2.08 (a 2.03 é passivo financeiro); demais usam 2.03
    fin_mask  = df.get("is_financeiro", pd.Series(False, index=df.index)).fillna(False).astype(bool)
    pl        = pl_nonfin.where(~fin_mask, pl_fin)
    df["pl"]  = pl

    df["disponibilidades"] = dispon
    df["divida_bruta"]     = div_bruta
    df["divida_liquida"]   = div_liq

    # --- Alíquota efetiva (sobre valores anualizados) ---
    ll_anualizado = df["ll_anualizado"]
    base_ir = ll_anualizado.abs() + ir * fator
    df["aliquota_efetiva"] = np.clip(
        np.where(base_ir > 0, (ir * fator) / base_ir, 0.34), 0, 0.50
    )

    df["nopat"]             = ebit * (1 - df["aliquota_efetiva"])
    cap_inv                 = pl.fillna(0) + div_liq
    df["capital_investido"] = cap_inv
    df["roic"]              = np.where(cap_inv.abs() > 1e-6, df["nopat"] / cap_inv, np.nan)
    # ROIC não se aplica a instituições financeiras (dívida é operacional)
    fin = df.get("is_financeiro", pd.Series(False, index=df.index)).fillna(False).astype(bool)
    df.loc[fin, "roic"] = np.nan

    df["market_cap"] = np.nan
    df["ev"]         = np.nan
    df["ev_ebit"]    = np.nan

    return df


# ---------------------------------------------------------------------------
# 6. Ranking por trimestre (ROIC — referência; ranking composto vem depois)
# ---------------------------------------------------------------------------

def _rankear(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ROIC: maior é melhor → rank 1 = maior ROIC
    df["rank_roic"] = (
        df[df["roic"] > 0]        # exclui ROIC negativos do ranking
          .groupby("DT_FIM_EXERC")["roic"]
          .rank(ascending=False, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")

    return df.sort_values(["DT_FIM_EXERC", "rank_roic"])


# ---------------------------------------------------------------------------
# 7. Pipeline principal
# ---------------------------------------------------------------------------

def processar_ano(ano: int) -> pd.DataFrame:
    """
    Pipeline completo para um ano.
    Baixa ITR (Q1–Q3) + DFP (Q4), filtra financeiras, anualiza fluxos (EBIT, EBITDA,
    Receita, LL), anexa qt_acoes (CVM) e SETOR_ATIV, e retorna ranking trimestral.

    Para vários anos:
        frames = [processar_ano(a) for a in range(2015, 2026)]
        historico = pd.concat(frames, ignore_index=True)
    """
    print(f"\n{'='*55}")
    print(f"Processando {ano}")
    print(f"{'='*55}")

    # Download
    dados = baixar_dados(ano)
    if not dados:
        return pd.DataFrame()

    # Cadastro: SETOR_ATIV (financeiras agora são INCLUÍDAS e marcadas, não excluídas)
    print("Carregando cadastro (setores)...")
    cad = _carregar_cadastro()

    # Limpeza
    print("Limpando dados...")
    dre = _limpar(dados["dre"])
    bpa = _limpar(dados["bpa"])
    bpp = _limpar(dados["bpp"])
    dfc = _limpar(dados["dfc"]) if "dfc" in dados else pd.DataFrame()

    # Pivotamento
    print("Pivotando contas...")
    df_dre = _pivotar_dre(dre)
    df_bpa = _pivotar_balanco(bpa, CONTAS_BPA)
    df_bpp = _pivotar_balanco(bpp, CONTAS_BPP)
    df_da  = _pivotar_da(dfc) if not dfc.empty else \
             pd.DataFrame(columns=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC", "DT_INI_EXERC", "d_a"])

    # Merge sem DENOM_CIA na chave (nomes divergentes entre demonstrativos e
    # entre ITR/DFP geravam linhas duplicadas). Chave = CNPJ + data (+ DT_INI nos fluxos).
    chave_fluxo = ["CNPJ_CIA", "DT_FIM_EXERC", "DT_INI_EXERC"]
    chave_bal   = ["CNPJ_CIA", "DT_FIM_EXERC"]
    base = df_dre.merge(df_da, on=chave_fluxo, how="left")
    base = (
        base
        .merge(df_bpa, on=chave_bal, how="outer")
        .merge(df_bpp, on=chave_bal, how="outer")
    )

    # Nome canônico por CNPJ (um DENOM_CIA por empresa)
    denom_map = (
        pd.concat([dre[["CNPJ_CIA", "DENOM_CIA"]],
                   bpa[["CNPJ_CIA", "DENOM_CIA"]],
                   bpp[["CNPJ_CIA", "DENOM_CIA"]]], ignore_index=True)
          .dropna().drop_duplicates("CNPJ_CIA")
    )
    base = base.merge(denom_map, on="CNPJ_CIA", how="left")

    # qt_acoes (CVM) — pontual, por CNPJ + data
    if "qt_acoes" in dados:
        base = base.merge(dados["qt_acoes"], on=["CNPJ_CIA", "DT_FIM_EXERC"], how="left")
    else:
        base["qt_acoes"] = np.nan

    # SETOR_ATIV + marca financeiras (não exclui; métricas inaplicáveis viram NaN)
    base = base.merge(cad, on="CNPJ_CIA", how="left")
    base["is_financeiro"] = base["SETOR_ATIV"].isin(SETORES_FINANCEIROS)

    # Desduplicação final: 1 linha por (CNPJ, DT_FIM_EXERC), mantendo a mais
    # completa (mais colunas preenchidas) — rede de segurança contra resíduos.
    base = (
        base.assign(_n=base.notna().sum(axis=1))
            .sort_values("_n", ascending=False)
            .drop_duplicates(["CNPJ_CIA", "DT_FIM_EXERC"], keep="first")
            .drop(columns="_n")
    )

    trimestres = sorted(base["DT_FIM_EXERC"].dropna().unique())
    print(f"  {base['CNPJ_CIA'].nunique():,} empresas | "
          f"{len(trimestres)} datas: {[str(t)[:10] for t in trimestres]}")

    # Cálculo e ranking
    print("Calculando indicadores (fluxos anualizados)...")
    resultado = _calcular(base)
    resultado = _rankear(resultado)

    # Colunas finais
    cols = [
        "CNPJ_CIA", "DENOM_CIA", "SETOR_ATIV", "is_financeiro", "DT_FIM_EXERC", "DT_INI_EXERC", "meses_periodo",
        "receita", "receita_anualizada",
        "ebit", "ebit_anualizado", "d_a", "d_a_anualizado", "ebitda", "nopat",
        "ll", "ll_anualizado",
        "disponibilidades", "divida_bruta", "divida_liquida",
        "pl", "capital_investido", "qt_acoes",
        "roic", "rank_roic",
        "market_cap", "ev", "ev_ebit",
    ]
    cols_presentes = [c for c in cols if c in resultado.columns]
    resultado = resultado[cols_presentes]

    print(f"✅ {ano} concluído: {len(resultado):,} linhas")
    return resultado


In [3]:
import pandas as pd
cad = pd.read_csv(
    "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv",
    sep=";", encoding="latin1", dtype=str
)
cad.columns = cad.columns.str.strip()
print(cad["SETOR_ATIV"].value_counts().head(20))

KeyboardInterrupt: 

In [ ]:
# Um ano
#df_2023 = processar_ano(2023)

# Vários anos empilhados (CAGR 5a passa a existir a partir de 2020)
frames = [processar_ano(a) for a in range(2015, 2026)]
historico = pd.concat(frames, ignore_index=True)
historico.to_csv("ranking_historico.csv", index=False, encoding="utf-8-sig")



Processando 2015
  Baixando ITR 2015... ✓
    ⚠ itr_cia_aberta_composicao_capital_2015.csv não encontrado — qt_acoes ficará NaN
  Baixando DFP 2015... ✓
    ⚠ dfp_cia_aberta_composicao_capital_2015.csv não encontrado — qt_acoes ficará NaN
Carregando cadastro (setores)...
Limpando dados...
Pivotando contas...
  355 empresas | 4 datas: ['2015-03-31', '2015-06-30', '2015-09-30', '2015-12-31']
Calculando indicadores (fluxos anualizados)...
✅ 2015 concluído: 1,317 linhas

Processando 2016
  Baixando ITR 2016... ✓
    ⚠ itr_cia_aberta_composicao_capital_2016.csv não encontrado — qt_acoes ficará NaN
  Baixando DFP 2016... ✓
    ⚠ dfp_cia_aberta_composicao_capital_2016.csv não encontrado — qt_acoes ficará NaN
Carregando cadastro (setores)...
Limpando dados...
Pivotando contas...
  352 empresas | 4 datas: ['2016-03-31', '2016-06-30', '2016-09-30', '2016-12-31']
Calculando indicadores (fluxos anualizados)...
✅ 2016 concluído: 1,294 linhas

Processando 2017
  Baixando ITR 2017... ✓
    ⚠ itr_cia

Tendo ranking_historico.csv baixado, rodar a partir do código abaixo

In [ ]:
import warnings
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from rapidfuzz import process, fuzz
import time
 
warnings.filterwarnings("ignore")

historico = pd.read_csv('ranking_historico.csv')

In [ ]:
# ---------------------------------------------------------------------------
# 3. Preço histórico e shares via yfinance (para Market Cap e EV)
# ---------------------------------------------------------------------------
 
def _buscar_preco_shares(ticker: str, datas: list) -> pd.DataFrame:
    """
    Busca o preço de fechamento ajustado e quantidade de ações via yfinance
    para as datas dos demonstrativos.
 
    Retorna DataFrame com [DT_FIM_EXERC, preco_fechamento, shares_outstanding].
    """
    try:
        t = yf.Ticker(f"{ticker}.SA")
        info = t.info
 
        # Shares: prefere fast_info, cai para info
        shares = (
            getattr(t.fast_info, "shares", None)
            or info.get("sharesOutstanding")
        )
 
        # Histórico de preços — pega intervalo que cobre todas as datas
        datas_dt = pd.to_datetime(datas)
        start = datas_dt.min() - pd.DateOffset(days=10)
        end   = datas_dt.max() + pd.DateOffset(days=10)
 
        hist = t.history(start=start, end=end, interval="1mo", auto_adjust=True)
        if hist.empty:
            return pd.DataFrame()
 
        hist = hist[["Close"]].reset_index()
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)
        hist["shares"] = shares
 
        rows = []
        for data in datas_dt:
            # Preço mais próximo da data de fim do exercício
            diff = (hist["Date"] - data).abs()
            idx = diff.idxmin()
            rows.append({
                "DT_FIM_EXERC": data,
                "preco_fechamento": hist.loc[idx, "Close"],
                "shares_outstanding": shares,
            })
 
        return pd.DataFrame(rows)
 
    except Exception as e:
        print(f"    ⚠ yfinance ({ticker}): {e}")
        return pd.DataFrame()

# ---------------------------------------------------------------------------
# 4. Cálculo dos indicadores
# ---------------------------------------------------------------------------
 
def _calcular_indicadores(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Calcula EBIT, Net Debt, Market Cap, EV e ROIC a partir das colunas extraídas.
 
    Entradas esperadas (colunas):
        ebit, caixa, aplic_cp, divida_cp, divida_lp, pl, ativo_total,
        ir_csll, ll, preco_fechamento, shares_outstanding
    """
    df = df.copy()
 
    # --- Dívida Líquida ---
    df["disponibilidades"] = df.get("caixa", 0).fillna(0) + df.get("aplic_cp", 0).fillna(0)
    df["divida_bruta"]     = df.get("divida_cp", 0).fillna(0) + df.get("divida_lp", 0).fillna(0)
    df["divida_liquida"]   = df["divida_bruta"] - df["disponibilidades"]
 
    # --- Market Cap ---
    if "preco_fechamento" in df.columns and "shares_outstanding" in df.columns:
        df["market_cap"] = df["preco_fechamento"] * df["shares_outstanding"].fillna(0)
    else:
        df["market_cap"] = np.nan
 
    # --- EV ---
    df["ev"] = df["market_cap"] + df["divida_liquida"]
 
    # --- Alíquota Efetiva de IR/CSLL ---
    # IR e CSLL costumam vir negativos na DRE (despesa)
    df["ir_csll"]  = df.get("ir_csll", pd.Series(0, index=df.index)).fillna(0).abs()
    
    df["ebt"] = (
        df.get("ebit", pd.Series(np.nan, index=df.index)) +
        df.get("resultado_financeiro", pd.Series(0, index=df.index)).fillna(0)
        )
 
    # Evita divisão por zero
    with np.errstate(divide="ignore", invalid="ignore"):
        df["aliquota_efetiva"] = np.where(
            df.get("ll", pd.Series(np.nan, index=df.index)).abs() > 0,
            df["ir_csll"] / (df.get("ll", pd.Series(np.nan, index=df.index)).abs() + df["ir_csll"]),
            0.34,   # alíquota padrão BR se não conseguir calcular
        )
        df["aliquota_efetiva"] = df["aliquota_efetiva"].clip(0, 0.50)
#     df["aliquota_efetiva"] = np.where(
#     df.get("ll", pd.Series(np.nan, index=df.index)).abs() > 0,
#     df["ir_csll"] / (df.get("ll", pd.Series(np.nan, index=df.index)).abs() + df["ir_csll"]),
#     0.34,
# )

    # Linha do NOPAT
    df["nopat"] = df.get("ebit", pd.Series(np.nan, index=df.index)) * (1 - df["aliquota_efetiva"])

    # Linha do capital investido
    df["capital_investido"] = df.get("pl", pd.Series(np.nan, index=df.index)).fillna(0) + df["divida_liquida"]

    # Linha do ROIC
    df["roic"] = np.where(
        df["capital_investido"] != 0,
        df["nopat"] / df["capital_investido"],
        np.nan,
    )

    # Linha do EV/EBIT
    df["ev_ebit"] = np.where(
        df.get("ebit", pd.Series(np.nan, index=df.index)) != 0,
        df["ev"] / df.get("ebit", pd.Series(np.nan, index=df.index)),
        np.nan,
    )
 
    df.insert(0, "ticker", ticker)
    return df


In [ ]:
"""
adicionar_ev.py
===============
Adiciona ticker, preço (yfinance), market_cap, ev e ev_ebit ao DataFrame do
itr_ranking.

Mudança importante:
    - O número de ações agora vem da CVM (coluna qt_acoes = QT_ACAO_TOTAL_CAP_INTEGR),
      que reflete o histórico trimestral real, e NÃO mais o sharesOutstanding atual
      do yfinance.
    - market_cap = preco (yfinance, na data do balanço) × qt_acoes (CVM).
    - O ranking deixou de ser aqui: rank_composto é calculado em selecionar_trimestre.
"""

import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from rapidfuzz import process, fuzz


def _normalizar_escala_acoes(qt, pl, preco, limite_pb=0.05):
    """
    Corrige a escala de qt_acoes. A composição de capital da CVM às vezes
    informa as ações em MILHARES (o arquivo não tem coluna de escala), deixando
    qt ~1000x menor e inflando LPA/VPA/Graham. Detecta pelo P/B implícito
    (market_cap / PL): enquanto < limite_pb, multiplica qt por 1000 (até 2x).
    Só ajusta linhas com PL > 0 e preço disponível.
    """
    qt    = np.asarray(qt,    dtype="float64").copy()
    pl    = np.asarray(pl,    dtype="float64")
    preco = np.asarray(preco, dtype="float64")
    for _ in range(2):
        pb = np.where(
            (pl > 0) & np.isfinite(preco) & np.isfinite(qt) & (qt > 0),
            (preco * qt) / pl, np.nan,
        )
        corrige = np.isfinite(pb) & (pb < limite_pb)
        qt = np.where(corrige, qt * 1000.0, qt)
    return qt


# ---------------------------------------------------------------------------
# Mapeamento CNPJ → ticker via brapi.dev + fuzzy match
# ---------------------------------------------------------------------------

def _buscar_todos_tickers_brapi() -> pd.DataFrame:
    url = "https://brapi.dev/api/quote/list"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        stocks = r.json().get("stocks", [])
        df = pd.DataFrame(stocks)[["stock", "name"]].copy()
        df.columns = ["ticker", "nome"]
        df["ticker"] = df["ticker"].str.upper().str.strip()
        df["nome"]   = df["nome"].str.upper().str.strip()
        df = df[df["ticker"].str.match(r'^[A-Z]{4}\d{1,2}$')]
        print(f"  brapi.dev: {len(df):,} tickers carregados")
        return df.drop_duplicates("ticker").reset_index(drop=True)
    except Exception as e:
        print(f"  ✗ Erro brapi.dev: {e}")
        return pd.DataFrame(columns=["ticker", "nome"])


def _normalizar_nome(nome: str) -> str:
    remover = [
        "S.A.", "S/A", "SA", "LTDA", "LTDA.", "S.A", "/SA",
        "CIA.", "CIA", "COMPANHIA", "PARTICIPACOES", "PARTICIPAÇÕES",
        "HOLDING", "GROUP", "BRASIL", "DO BRASIL",
        "EM RECUPERACAO JUDICIAL", "EM LIQUIDACAO EXTRAJUDICIAL",
    ]
    nome = nome.upper()
    for t in remover:
        nome = nome.replace(t, "")
    return " ".join(nome.split())


def _baixar_mapa_cnpj_ticker(df_ranking: pd.DataFrame) -> pd.DataFrame:
    empresas = (
        df_ranking[["CNPJ_CIA", "DENOM_CIA"]]
        .drop_duplicates("CNPJ_CIA")
        .dropna(subset=["DENOM_CIA"])
        .copy()
    )
    print(f"  Empresas para mapear: {len(empresas):,}")

    df_brapi = _buscar_todos_tickers_brapi()
    if df_brapi.empty:
        return pd.DataFrame(columns=["CNPJ_CIA", "TCKR"])

    nomes_brapi_norm = df_brapi["nome"].apply(_normalizar_nome).tolist()
    tickers_brapi    = df_brapi["ticker"].tolist()

    mapa, score = {}, {}
    for nome in empresas["DENOM_CIA"].unique():
        nome_norm = _normalizar_nome(nome)
        match = process.extractOne(
            nome_norm, nomes_brapi_norm, scorer=fuzz.token_sort_ratio
        )
        if match and match[1] >= 72:
            mapa[nome]  = tickers_brapi[nomes_brapi_norm.index(match[0])]
            score[nome] = match[1]
        else:
            mapa[nome], score[nome] = None, 0

    empresas = empresas.copy()
    empresas["TCKR"]   = empresas["DENOM_CIA"].map(mapa)
    empresas["_score"] = empresas["DENOM_CIA"].map(score)
    empresas = empresas.dropna(subset=["TCKR"])

    # 1 ticker → 1 CNPJ (o de maior score): elimina falsos positivos em que
    # subsidiárias/holdings de nome parecido roubam o ticker da principal
    # (ex.: Energisa/Neoenergia/Rio Energy disputando ENGI11).
    antes = len(empresas)
    empresas = empresas.sort_values("_score", ascending=False).drop_duplicates("TCKR", keep="first")
    print(f"  Mapeadas: {len(empresas):,}/{antes:,} (resolvidos {antes-len(empresas):,} "
          f"conflitos de ticker; demais CNPJs não listados na B3 — esperado)")

    return empresas[["CNPJ_CIA", "TCKR"]]


# ---------------------------------------------------------------------------
# Preços históricos via yfinance (apenas preço — ações vêm da CVM)
# ---------------------------------------------------------------------------

def _buscar_precos_ticker(ticker: str, datas) -> pd.DataFrame:
    datas_dt = pd.to_datetime(datas)
    try:
        t = yf.Ticker(f"{ticker}.SA")
        hist = t.history(
            start=datas_dt.min() - pd.DateOffset(days=15),
            end=datas_dt.max()   + pd.DateOffset(days=15),
            interval="1mo",
            auto_adjust=True,
        )
        if hist.empty:
            return pd.DataFrame()

        hist = hist[["Close"]].reset_index()
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)

        rows = []
        for data in datas_dt:
            idx = (hist["Date"] - data).abs().idxmin()
            rows.append({
                "DT_FIM_EXERC": data,
                "preco":        hist.loc[idx, "Close"],
            })
        return pd.DataFrame(rows)
    except Exception:
        return pd.DataFrame()


# ---------------------------------------------------------------------------
# Pipeline principal
# ---------------------------------------------------------------------------

def adicionar_ev(df_ranking: pd.DataFrame, delay: float = 0.3) -> pd.DataFrame:
    """
    Adiciona ticker, preco, market_cap, ev e ev_ebit.
    market_cap = preco (yfinance) × qt_acoes (CVM, já presente no DataFrame).
    """
    df = df_ranking.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])

    # --- Mapeamento CNPJ → ticker ---
    print("Construindo mapeamento CNPJ → ticker...")
    mapa = _baixar_mapa_cnpj_ticker(df)
    if mapa.empty:
        print("✗ Sem mapeamento. Abortando.")
        return df

    df = df.merge(mapa, on="CNPJ_CIA", how="left")
    print(f"  {df['TCKR'].notna().sum():,} linhas com ticker | "
          f"{df['TCKR'].isna().sum():,} sem ticker")

    # --- Busca preços: um ticker de cada vez ---
    tickers_unicos = df["TCKR"].dropna().unique()
    print(f"\nBuscando preços para {len(tickers_unicos):,} tickers...")

    frames_precos = []
    for i, ticker in enumerate(tickers_unicos):
        datas = df.loc[df["TCKR"] == ticker, "DT_FIM_EXERC"].dropna().unique()
        df_p  = _buscar_precos_ticker(ticker, datas)
        if not df_p.empty:
            df_p["TCKR"] = ticker
            frames_precos.append(df_p)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(tickers_unicos)} ({len(frames_precos)} com dados)...")
        time.sleep(delay)

    print(f"  Preços obtidos: {len(frames_precos):,} tickers")

    if not frames_precos:
        print("✗ Nenhum preço obtido.")
        return df.drop(columns=["TCKR"], errors="ignore")

    # --- Merge de preços por (TCKR, DT_FIM_EXERC) ---
    df_precos = pd.concat(frames_precos, ignore_index=True)
    df_precos["DT_FIM_EXERC"] = pd.to_datetime(df_precos["DT_FIM_EXERC"])

    df = df.merge(
        df_precos[["TCKR", "DT_FIM_EXERC", "preco"]],
        on=["TCKR", "DT_FIM_EXERC"],
        how="left",
    )
    df = df.rename(columns={"TCKR": "ticker"})

    # --- Corrige escala de qt_acoes (composição às vezes em MILHARES) ---
    pl_col = df["pl"] if "pl" in df.columns else np.nan
    df["qt_acoes"] = _normalizar_escala_acoes(df["qt_acoes"], pl_col, df["preco"])

    # --- Indicadores de mercado (ações = qt_acoes da CVM) ---
    df["market_cap"] = df["preco"] * df["qt_acoes"]
    df["ev"]         = df["market_cap"] + df["divida_liquida"]
    df["ev_ebit"]    = np.where(
        df["ebit_anualizado"].notna() & (df["ebit_anualizado"] != 0),
        df["ev"] / df["ebit_anualizado"],
        np.nan,
    )
    # EV/EV-EBIT não se aplicam a instituições financeiras (market_cap permanece)
    if "is_financeiro" in df.columns:
        fin = df["is_financeiro"].fillna(False).astype(bool)
        df.loc[fin, ["ev", "ev_ebit"]] = np.nan

    # --- Reordena: ticker logo após DENOM_CIA ---
    cols = df.columns.tolist()
    if "ticker" in cols:
        cols.remove("ticker")
        idx = cols.index("DENOM_CIA")
        cols = cols[:idx + 1] + ["ticker"] + cols[idx + 1:]
        df = df[cols]

    preenchidos = df["market_cap"].notna().sum()
    print(f"\n✅ market_cap preenchido: {preenchidos:,}/{len(df):,} "
          f"({100*preenchidos/len(df):.1f}%)")
    return df


In [ ]:
"""
calcular_metricas.py
====================
Calcula métricas fundamentalistas por linha (trimestre × empresa) sobre o
DataFrame que já passou por processar_ano + adicionar_ev.

Inclui:
    ROE, Margem Líquida, P/L, DL/EBITDA
    LPA, VPA, Preço Justo de Graham e Margem de Segurança
    CAGR de Lucro e Receita 5 anos (dezembro X vs dezembro X-5)
    Selic (BCB série 432) por trimestre
    Flags de descarte (booleanas — aplicadas depois em selecionar_trimestre)

Todos os itens de fluxo usam os valores ANUALIZADOS (ll_anualizado,
receita_anualizada, ebitda); balanço (pl, divida_liquida) é pontual.
"""

import time
import requests
import datetime as _dt
import numpy as np
import pandas as pd

# Setores tratados como infraestrutura → isentos do filtro DL/EBITDA > 3
SETORES_INFRA = {"Petróleo e Gás"}

# Limiares dos filtros de descarte
LIMIAR_PL          = 15.0
LIMIAR_DL_EBITDA   = 3.0
LIMIAR_MARGEM_LIQ  = 0.13
MARGEM_SELIC       = 0.02   # ROE precisa superar Selic - 2 p.p.


# ---------------------------------------------------------------------------
# Selic (BCB série 432 — meta Selic, % a.a.)
# ---------------------------------------------------------------------------

def _selic_serie() -> pd.DataFrame:
    """
    Baixa a série 432 do SGS/BCB (últimos ~10 anos) e devolve [data, selic]
    com selic em fração decimal (ex.: 0.1375). Ordenada por data.
    """
    di  = (_dt.date.today() - _dt.timedelta(days=365 * 10)).strftime("%d/%m/%Y")
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"
           f"?formato=json&dataInicial={di}")
    for tentativa in range(3):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            s = pd.DataFrame(r.json())
            s["data"]  = pd.to_datetime(s["data"], format="%d/%m/%Y")
            s["selic"] = pd.to_numeric(s["valor"], errors="coerce") / 100
            return s[["data", "selic"]].dropna().sort_values("data").reset_index(drop=True)
        except Exception as e:
            print(f"  ⚠ Selic tentativa {tentativa+1}/3: {e}")
            time.sleep(2)
    print("  ⚠ Não foi possível obter a Selic — coluna ficará NaN")
    return pd.DataFrame(columns=["data", "selic"])


# ---------------------------------------------------------------------------
# CAGR 5 anos (dezembro X vs dezembro X-5)
# ---------------------------------------------------------------------------

def _cagr_5anos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula CAGR de lucro e receita comparando dezembro do ano X com dezembro
    do ano X-5. Retorna [CNPJ_CIA, data_cagr, cagr_lucro, cagr_receita], com
    data_cagr = 31/dez do ano X — para propagar via merge_asof (carry-forward).
    """
    dez = df[df["DT_FIM_EXERC"].dt.month == 12].copy()
    if dez.empty:
        return pd.DataFrame(columns=["CNPJ_CIA", "data_cagr", "cagr_lucro", "cagr_receita"])

    dez["ano"] = dez["DT_FIM_EXERC"].dt.year
    atual = (
        dez[["CNPJ_CIA", "ano", "ll_anualizado", "receita_anualizada"]]
        .drop_duplicates(["CNPJ_CIA", "ano"])
    )
    base = atual.rename(columns={
        "ll_anualizado": "ll_base",
        "receita_anualizada": "rec_base",
        "ano": "ano_base",
    })
    base["ano"] = base["ano_base"] + 5   # ano_base + 5 = ano atual

    m = atual.merge(
        base[["CNPJ_CIA", "ano", "ll_base", "rec_base"]],
        on=["CNPJ_CIA", "ano"], how="left",
    )

    def _cagr(atual_v, base_v):
        ok = (base_v > 0) & (atual_v > 0)
        return np.where(ok, (atual_v / base_v) ** (1 / 5) - 1, np.nan)

    m["cagr_lucro"]   = _cagr(m["ll_anualizado"], m["ll_base"])
    m["cagr_receita"] = _cagr(m["receita_anualizada"], m["rec_base"])
    m["data_cagr"]    = pd.to_datetime(m["ano"].astype(str) + "-12-31")
    return m[["CNPJ_CIA", "data_cagr", "cagr_lucro", "cagr_receita"]]


# ---------------------------------------------------------------------------
# Pipeline de métricas
# ---------------------------------------------------------------------------

def calcular_metricas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])

    ll  = df.get("ll_anualizado",      pd.Series(np.nan, index=df.index))
    rec = df.get("receita_anualizada", pd.Series(np.nan, index=df.index))
    pl  = df.get("pl",                 pd.Series(np.nan, index=df.index))
    ebitda = df.get("ebitda",          pd.Series(np.nan, index=df.index))
    qt  = df.get("qt_acoes",           pd.Series(np.nan, index=df.index))
    mc  = df.get("market_cap",         pd.Series(np.nan, index=df.index))
    preco = df.get("preco",            pd.Series(np.nan, index=df.index))
    div_liq = df.get("divida_liquida", pd.Series(np.nan, index=df.index))

    # --- Indicadores fundamentalistas ---
    df["roe"]            = np.where(pl > 0, ll / pl, np.nan)
    df["margem_liquida"] = np.where(rec.abs() > 0, ll / rec, np.nan)
    df["p_l"]            = np.where(ll > 0, mc / ll, np.nan)
    df["dl_ebitda"]      = np.where(ebitda > 0, div_liq / ebitda, np.nan)
    # DL/EBITDA não se aplica a instituições financeiras
    _fin = df.get("is_financeiro", pd.Series(False, index=df.index)).fillna(False).astype(bool)
    df.loc[_fin, "dl_ebitda"] = np.nan

    # --- Graham ---
    df["lpa"] = np.where(qt > 0, ll / qt, np.nan)
    df["vpa"] = np.where(qt > 0, pl / qt, np.nan)
    df["preco_graham"] = np.where(
        (df["lpa"] > 0) & (df["vpa"] > 0),
        np.sqrt(22.5 * df["lpa"] * df["vpa"]),
        np.nan,
    )
    df["margem_seguranca"] = np.where(
        df["preco_graham"].notna() & (df["preco_graham"] > 0) & preco.notna(),
        (df["preco_graham"] - preco) / df["preco_graham"],
        np.nan,
    )

    # --- CAGR 5 anos: calculado em dezembro (Dez X vs Dez X-5) e propagado aos
    # trimestres seguintes via merge_asof (carry-forward; usa só dezembros passados) ---
    cagr = _cagr_5anos(df).dropna(subset=["data_cagr"]).sort_values("data_cagr")
    df = df.sort_values("DT_FIM_EXERC")
    if not cagr.empty:
        df = pd.merge_asof(
            df, cagr, left_on="DT_FIM_EXERC", right_on="data_cagr",
            by="CNPJ_CIA", direction="backward",
        ).drop(columns=["data_cagr"])
    else:
        df["cagr_lucro"] = np.nan
        df["cagr_receita"] = np.nan

    # --- Selic por trimestre (valor vigente até DT_FIM_EXERC) ---
    selic = _selic_serie()
    if not selic.empty:
        df = df.sort_values("DT_FIM_EXERC")
        df = pd.merge_asof(
            df, selic.rename(columns={"data": "DT_FIM_EXERC"}),
            on="DT_FIM_EXERC", direction="backward",
        )
    else:
        df["selic"] = np.nan

    # --- Flags de descarte (booleanas; não removem nada ainda) ---
    df["flag_prejuizo"]    = ll <= 0
    df["flag_pl_alto"]     = df["p_l"] > LIMIAR_PL
    eh_infra               = df.get("SETOR_ATIV", pd.Series(index=df.index)).isin(SETORES_INFRA)
    df["flag_alavancagem"] = (df["dl_ebitda"] > LIMIAR_DL_EBITDA) & (~eh_infra)
    df["flag_cagr_neg"]    = df["cagr_lucro"].notna() & (df["cagr_lucro"] < 0)
    df["flag_margem_baixa"] = df["margem_liquida"] < LIMIAR_MARGEM_LIQ
    df["flag_roe_baixo"]   = df["roe"] < (df["selic"] - MARGEM_SELIC)

    return df


In [8]:
"""
ranking_composto.py
===================
Ranking composto de pesos iguais e seleção por trimestre/setor.

rank_composto = 0.25*rank_roic + 0.25*rank_graham
              + 0.25*rank_dl_ebitda + 0.25*rank_cagr_lucro
(menor rank_composto = melhor)

Participam do ranking apenas empresas com:
    ROIC > 0, margem de segurança calculável e DL/EBITDA disponível.
Os ranks são calculados por trimestre (groupby DT_FIM_EXERC) entre as empresas
que sobreviveram aos filtros de descarte.
"""

import numpy as np
import pandas as pd

FLAGS_DESCARTE = [
    "flag_prejuizo", "flag_pl_alto", "flag_alavancagem",
    "flag_cagr_neg", "flag_margem_baixa", "flag_roe_baixo",
]


def _aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    """Remove linhas com qualquer flag de descarte verdadeira."""
    flags = [f for f in FLAGS_DESCARTE if f in df.columns]
    if not flags:
        return df.copy()
    # robusto a flags carregadas de CSV como strings "True"/"False"
    descartar = (
        df[flags].astype(str)
                 .apply(lambda s: s.str.strip().str.lower().isin(["true", "1", "1.0"]))
                 .any(axis=1)
    )
    return df[~descartar].copy()


def _aplicar_liquidez(df: pd.DataFrame, df_volumes, trimestre,
                      liquidez_min: float = 1_000_000) -> pd.DataFrame:
    """
    Mantém apenas tickers com liquidez média diária (R$) >= liquidez_min no
    trimestre. df_volumes: [ticker, trimestre, vol_medio_diario]. Se df_volumes
    for None ou liquidez_min <= 0, não filtra. Acrescenta vol_medio_diario ao
    resultado quando o filtro é aplicado.
    """
    if df_volumes is None or not liquidez_min or liquidez_min <= 0:
        return df
    tri = str(pd.Period(trimestre, freq="Q"))
    vol = (
        df_volumes[df_volumes["trimestre"] == tri][["ticker", "vol_medio_diario"]]
        .drop_duplicates("ticker")
    )
    out = df.merge(vol, on="ticker", how="left")
    return out[out["vol_medio_diario"].fillna(0) >= liquidez_min].copy()


def _calcular_rank_composto(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula os 4 ranks e o rank_composto por trimestre, apenas entre empresas
    elegíveis (ROIC > 0, margem de segurança calculável, DL/EBITDA disponível).
    """
    elegivel = (
        (df["roic"] > 0)
        & df["margem_seguranca"].notna()
        & df["dl_ebitda"].notna()
    )
    d = df[elegivel].copy()
    if d.empty:
        for c in ["rank_roic", "rank_graham", "rank_dl_ebitda",
                  "rank_cagr_lucro", "rank_composto"]:
            d[c] = pd.Series(dtype="float64")
        return d

    g = d.groupby("DT_FIM_EXERC")
    d["rank_roic"]       = g["roic"].rank(ascending=False, method="min")
    d["rank_graham"]     = g["margem_seguranca"].rank(ascending=False, method="min")
    d["rank_dl_ebitda"]  = g["dl_ebitda"].rank(ascending=True, method="min")
    # CAGR só existe a partir de 2020 → ausentes vão para o fim do ranking
    d["rank_cagr_lucro"] = g["cagr_lucro"].rank(
        ascending=False, method="min", na_option="bottom"
    )

    d["rank_composto"] = (
        0.25 * d["rank_roic"]
        + 0.25 * d["rank_graham"]
        + 0.25 * d["rank_dl_ebitda"]
        + 0.25 * d["rank_cagr_lucro"]
    )
    return d


def selecionar_trimestre(df: pd.DataFrame, trimestre, top_n: int = 15,
                         modo: str = "global", df_volumes: pd.DataFrame = None,
                         liquidez_min: float = 1_000_000) -> pd.DataFrame:
    """
    Seleciona as melhores empresas de um trimestre pelo rank_composto.

    Parâmetros:
        df        : DataFrame vindo de calcular_metricas.
        trimestre : período no formato 'AAAAQT' (ex.: '2023Q4') ou data.
        top_n     : quantas empresas retornar (por setor ou no total).
        modo      : 'global'      → top_n geral
                    'por_setor'   → top_n por SETOR_ATIV
                    '1_por_setor' → melhor de cada setor

    Retorna DataFrame ordenado por rank_composto (menor = melhor).
    """
    df = df.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])

    alvo = pd.Period(trimestre, freq="Q")
    mask = df["DT_FIM_EXERC"].dt.to_period("Q") == alvo
    sub = df[mask]
    if sub.empty:
        print(f"⚠ Nenhuma linha para o trimestre {trimestre}.")
        return sub

    # 1) descarte  2) liquidez mínima  3) ranking entre os sobreviventes
    sub = _aplicar_filtros(sub)
    sub = _aplicar_liquidez(sub, df_volumes, trimestre, liquidez_min)
    sub = _calcular_rank_composto(sub)
    sub = sub.sort_values("rank_composto").reset_index(drop=True)

    if modo == "global":
        return sub.head(top_n)
    if modo == "por_setor":
        return (
            sub.groupby("SETOR_ATIV", group_keys=False)
               .head(top_n)
               .sort_values(["SETOR_ATIV", "rank_composto"])
        )
    if modo == "1_por_setor":
        return (
            sub.groupby("SETOR_ATIV", group_keys=False)
               .head(1)
               .sort_values("rank_composto")
        )
    raise ValueError(f"modo inválido: {modo!r} (use 'global', 'por_setor' ou '1_por_setor')")



# ---------------------------------------------------------------------------
# Ranking separado para BANCOS / setor financeiro
# (ROIC, DL/EBITDA e EV não se aplicam; usa-se ROE + Graham + P/L + CAGR —
#  como analistas avaliam instituições financeiras)
# ---------------------------------------------------------------------------

def _calcular_rank_banco(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ranking de financeiras por trimestre: pesos iguais em
    ROE (maior melhor), Margem de Segurança/Graham (maior), P/L (menor),
    CAGR de lucro 5a (maior). Elegíveis: is_financeiro, ROE>0,
    margem de segurança calculável e P/L>0.
    """
    fin = df.get("is_financeiro", pd.Series(False, index=df.index)).fillna(False).astype(bool)
    elegivel = (
        fin
        & (df["roe"] > 0)
        & df["margem_seguranca"].notna()
        & (df["p_l"] > 0)
    )
    d = df[elegivel].copy()
    if d.empty:
        for c in ["rank_roe", "rank_graham", "rank_pl", "rank_cagr_lucro", "rank_banco"]:
            d[c] = pd.Series(dtype="float64")
        return d

    g = d.groupby("DT_FIM_EXERC")
    d["rank_roe"]        = g["roe"].rank(ascending=False, method="min")
    d["rank_graham"]     = g["margem_seguranca"].rank(ascending=False, method="min")
    d["rank_pl"]         = g["p_l"].rank(ascending=True, method="min")
    d["rank_cagr_lucro"] = g["cagr_lucro"].rank(ascending=False, method="min", na_option="bottom")
    d["rank_banco"] = (
        0.25 * d["rank_roe"]
        + 0.25 * d["rank_graham"]
        + 0.25 * d["rank_pl"]
        + 0.25 * d["rank_cagr_lucro"]
    )
    return d


def selecionar_bancos(df: pd.DataFrame, trimestre, top_n: int = 15,
                      modo: str = "global", df_volumes: pd.DataFrame = None,
                      liquidez_min: float = 1_000_000) -> pd.DataFrame:
    """
    Seleciona as melhores instituições financeiras de um trimestre pelo
    rank_banco (ROE + Graham + P/L + CAGR). Mesmos modos de selecionar_trimestre:
    'global', 'por_setor' (SETOR_ATIV) e '1_por_setor'.
    """
    df = df.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])
    alvo = pd.Period(trimestre, freq="Q")
    sub = df[df["DT_FIM_EXERC"].dt.to_period("Q") == alvo]
    if sub.empty:
        print(f"⚠ Nenhuma linha para o trimestre {trimestre}.")
        return sub

    sub = _aplicar_liquidez(sub, df_volumes, trimestre, liquidez_min)
    sub = _calcular_rank_banco(sub).sort_values("rank_banco").reset_index(drop=True)

    if modo == "global":
        return sub.head(top_n)
    if modo == "por_setor":
        return (
            sub.groupby("SETOR_ATIV", group_keys=False)
               .head(top_n)
               .sort_values(["SETOR_ATIV", "rank_banco"])
        )
    if modo == "1_por_setor":
        return (
            sub.groupby("SETOR_ATIV", group_keys=False)
               .head(1)
               .sort_values("rank_banco")
        )
    raise ValueError(f"modo inválido: {modo!r} (use 'global', 'por_setor' ou '1_por_setor')")


In [ ]:
import pandas as pd
df_cad = pd.read_csv(
    "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv",
    sep=";", encoding="latin1", dtype=str
)
df_cad.columns = df_cad.columns.str.strip()
print(df_cad.columns.tolist())
print(df_cad.head(3))

['CNPJ_CIA', 'DENOM_SOCIAL', 'DENOM_COMERC', 'DT_REG', 'DT_CONST', 'DT_CANCEL', 'MOTIVO_CANCEL', 'SIT', 'DT_INI_SIT', 'CD_CVM', 'SETOR_ATIV', 'TP_MERC', 'CATEG_REG', 'DT_INI_CATEG', 'SIT_EMISSOR', 'DT_INI_SIT_EMISSOR', 'CONTROLE_ACIONARIO', 'TP_ENDER', 'LOGRADOURO', 'COMPL', 'BAIRRO', 'MUN', 'UF', 'PAIS', 'CEP', 'DDD_TEL', 'TEL', 'DDD_FAX', 'FAX', 'EMAIL', 'TP_RESP', 'RESP', 'DT_INI_RESP', 'LOGRADOURO_RESP', 'COMPL_RESP', 'BAIRRO_RESP', 'MUN_RESP', 'UF_RESP', 'PAIS_RESP', 'CEP_RESP', 'DDD_TEL_RESP', 'TEL_RESP', 'DDD_FAX_RESP', 'FAX_RESP', 'EMAIL_RESP', 'CNPJ_AUDITOR', 'AUDITOR']
             CNPJ_CIA                                       DENOM_SOCIAL  \
0  08.773.135/0001-00          2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL   
1  11.396.633/0001-87                        3A COMPANHIA SECURITIZADORA   
2  01.547.749/0001-16  521 PARTICIPAÇOES S.A. - EM LIQUIDAÇÃO EXTRAJU...   

                         DENOM_COMERC      DT_REG    DT_CONST   DT_CANCEL  \
0                     2W ECOBANK

In [ ]:
# Dados de mercado (ticker, preço, market_cap via qt_acoes da CVM, ev, ev_ebit)
historico_completo = adicionar_ev(historico)

# Métricas fundamentalistas + Graham + CAGR + Selic + flags de descarte
df_final = calcular_metricas(historico_completo)

# Desduplicação final: 1 linha por (CNPJ, DT_FIM_EXERC), mantendo a mais
# completa — elimina pares consolidado+individual que sobrevivem ao merge outer.
df_final["_completude"] = df_final.notna().sum(axis=1)
df_final = (
    df_final.sort_values("_completude", ascending=False)
            .drop_duplicates(subset=["CNPJ_CIA", "DT_FIM_EXERC"], keep="first")
            .drop(columns="_completude")
            .reset_index(drop=True)
)


Construindo mapeamento CNPJ → ticker...
  Empresas para mapear: 639
  brapi.dev: 1,272 tickers carregados
  Mapeadas: 265/303 (resolvidos 38 conflitos de ticker; demais CNPJs não listados na B3 — esperado)
  9,284 linhas com ticker | 7,941 sem ticker

Buscando preços para 265 tickers...
  50/265 (50 com dados)...
  100/265 (100 com dados)...
  150/265 (150 com dados)...
  200/265 (200 com dados)...


$PASS3.SA: possibly delisted; no price data found  (1mo 2019-12-16 00:00:00 -> 2026-01-15 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1576465200, endDate = 1768446000")


  250/265 (249 com dados)...
  Preços obtidos: 264 tickers

✅ market_cap preenchido: 5,700/17,225 (33.1%)
  ⚠ Selic tentativa 1/3: Expecting value: line 1 column 1 (char 0)
  ⚠ Selic tentativa 2/3: Expecting value: line 1 column 1 (char 0)


In [ ]:
df_final.to_csv("ranking.csv", index=False, encoding="utf-8-sig")

df_final


,CNPJ_CIA,DENOM_CIA,ticker,SETOR_ATIV,is_financeiro,DT_FIM_EXERC,DT_INI_EXERC,meses_periodo,receita,receita_anualizada,...,margem_seguranca,cagr_lucro,cagr_receita,selic,flag_prejuizo,flag_pl_alto,flag_alavancagem,flag_cagr_neg,flag_margem_baixa,flag_roe_baixo
0,59.105.999/0001-86,WHIRLPOOL S.A.,WHRL4,"Máquinas, Equipamentos, Veículos e Peças",False,2020-12-31,2020-01-01,12.0,9.258703e+09,9.258703e+09,...,-0.364831,0.196640,-0.002752,0.0200,False,False,False,False,True,False
1,33.611.500/0001-19,GERDAU S.A.,GGBR4,Metalurgia e Siderurgia,False,2025-12-31,2025-01-01,12.0,6.985853e+10,6.985853e+10,...,-0.068623,-0.098941,0.097792,0.1500,False,True,False,True,True,True
2,92.690.783/0001-09,METALURGICA GERDAU S.A.,GOAU4,Emp. Adm. Part. - Metalurgia e Siderurgia,False,2025-12-31,2025-01-01,12.0,6.985853e+10,6.985853e+10,...,0.688369,-0.099204,0.097792,0.1500,False,False,False,True,True,True
3,01.938.783/0001-11,CIA PARTICIPACOES ALIANCA DA BAHIA,PEAB4,Emp. Adm. Part. - Sem Setor Principal,False,2025-12-31,2025-01-01,12.0,1.172600e+08,1.172600e+08,...,0.119155,-0.408256,-0.083037,0.1500,False,True,False,True,False,True
4,97.837.181/0001-47,DEXCO S.A.,DXCO3,"Construção Civil, Mat. Constr. e Decoração",False,2025-12-31,2025-01-01,12.0,8.248752e+09,8.248752e+09,...,-0.646615,-0.326171,0.070059,0.1500,False,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17220,09.112.685/0001-32,OSX BRASIL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,Emp. Adm. Part. - Sem Setor Principal,False,2016-03-31,2016-01-01,3.0,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,NaN,False,False,False,False,False,False
17221,62.232.889/0001-90,BCO DAYCOVAL S.A.,NaN,Bancos,True,2021-12-31,2021-01-01,12.0,NaN,NaN,...,NaN,0.277538,NaN,0.0925,False,False,False,False,False,False
17222,15.073.274/0001-88,PPLA PARTICIPATIONS LTD.,NaN,NaN,False,2015-09-30,2015-01-01,9.0,-6.990140e+08,-9.320187e+08,...,NaN,NaN,NaN,NaN,True,False,False,False,False,False
17223,15.073.274/0001-88,PPLA PARTICIPATIONS LTD.,NaN,NaN,False,2015-12-31,2015-01-01,12.0,2.582960e+08,2.582960e+08,...,NaN,NaN,NaN,NaN,False,False,False,False,True,False


In [5]:
df_final = pd.read_csv("ranking.csv")


In [6]:
# ---------------------------------------------------------------------------
# Liquidez: volume financeiro médio diário por trimestre, para filtrar ações
# pouco negociadas na seleção. Baixado uma vez e cacheado em parquet.
# ---------------------------------------------------------------------------
import os
import pandas as pd
import yfinance as yf


def baixar_volumes(tickers: list, inicio: str = "2020-01-01") -> pd.DataFrame:
    """
    Retorna [ticker, trimestre, vol_medio_diario] - media diaria de
    (preco x volume) em cada trimestre, em R$/dia (liquidez).
    """
    tickers = sorted({t for t in tickers if isinstance(t, str) and t.strip()})
    tickers_sa = [f"{t}.SA" for t in tickers]
    print(f"Baixando volumes de {len(tickers_sa)} tickers...")

    dados = yf.download(
        tickers_sa, start=inicio, interval="1d",
        auto_adjust=True, progress=True,
    )

    # Colunas MultiIndex (campo, ticker) p/ varios tickers; simples p/ um so
    if isinstance(dados.columns, pd.MultiIndex):
        close, volume = dados["Close"].copy(), dados["Volume"].copy()
    else:
        close = dados[["Close"]].copy();  close.columns  = [tickers_sa[0]]
        volume = dados[["Volume"]].copy(); volume.columns = [tickers_sa[0]]
    close.columns  = [str(c).replace(".SA", "") for c in close.columns]
    volume.columns = [str(c).replace(".SA", "") for c in volume.columns]

    vol_fin = close * volume                       # volume financeiro diario (R$)
    vol_fin.index = pd.to_datetime(vol_fin.index)
    vol_tri = vol_fin.resample("QE").mean()        # media diaria por trimestre
    vol_tri.index = vol_tri.index.to_period("Q").astype(str)

    result = vol_tri.stack().reset_index()
    result.columns = ["trimestre", "ticker", "vol_medio_diario"]
    return result.dropna(subset=["vol_medio_diario"])


CAMINHO_VOL = "volumes_trimestrais.parquet"
if os.path.exists(CAMINHO_VOL):
    df_volumes = pd.read_parquet(CAMINHO_VOL)
    print(f"Volumes carregados do cache ({CAMINHO_VOL}): {len(df_volumes):,} linhas")
else:
    df_volumes = baixar_volumes(df_final["ticker"].dropna().unique().tolist())
    df_volumes.to_parquet(CAMINHO_VOL, index=False)
    print(f"Volumes salvos: {len(df_volumes):,} linhas")


Baixando volumes de 265 tickers...


[*********************100%***********************]  265 of 265 completed


Volumes salvos: 6,405 linhas


In [9]:
import pandas as pd

def extrair_top_n_empresas(dados, n=15, modo="global",
                           df_volumes=None, liquidez_min=1_000_000):
    """
    Monta uma tabela Trimestre × Top_1..Top_n com os tickers melhor ranqueados
    pelo rank_composto, chamando selecionar_trimestre em cada trimestre.

    `dados` pode ser o DataFrame df_final (recomendado, preserva os dtypes das
    flags) ou um caminho de CSV.
    """
    if isinstance(dados, str):
        dados = pd.read_csv(dados)
    df = dados.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])

    trimestres = sorted(df["DT_FIM_EXERC"].dt.to_period("Q").dropna().unique())
    linhas = {}
    for tri in trimestres:
        sel = selecionar_trimestre(df, str(tri), top_n=n, modo=modo,
                                   df_volumes=df_volumes, liquidez_min=liquidez_min)
        linhas[str(tri)] = sel["ticker"].head(n).tolist()

    pivot = pd.DataFrame.from_dict(linhas, orient="index")
    pivot.columns = [f"Top_{i+1}" for i in range(pivot.shape[1])]
    pivot.index.name = "Trimestre"
    return pivot

# Top 15 por trimestre pelo ranking composto, com liquidez mínima de R$1M/dia
resultado_df = extrair_top_n_empresas(df_final, n=15, modo="global",
                                      df_volumes=df_volumes, liquidez_min=1_000_000)
resultado_df.to_csv("melhores_por_trimestre.csv")
resultado_df


,Top_1,Top_2,Top_3,Top_4,Top_5,Top_6,Top_7,Top_8,Top_9,Top_10,Top_11,Top_12,Top_13,Top_14,Top_15
Trimestre,,,,,,,,,,,,,,,
2015Q1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2015Q2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2015Q3,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2015Q4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2016Q1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2016Q2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2016Q3,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2016Q4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2017Q1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [3]:
# ---------------------------------------------------------------------------
# Dados para o BACKTEST trimestral
# Para cada (trimestre de seleção, ticker) do melhores_por_trimestre.csv:
#   - market_cap : da CVM (ranking.csv / df_final), no trimestre da SELEÇÃO
#   - preço de entrada/saída: fechamento AJUSTADO (yfinance) no trimestre
#     SEGUINTE (holding) — os fundamentos de Q só são conhecidos depois, então
#     a posição é montada no trimestre seguinte (sem look-ahead)
#   - retorno do período, retorno do IBOV no período e liquidez (R$/dia)
# ---------------------------------------------------------------------------
import os
import pandas as pd
import yfinance as yf


def _close_diario_long(symbols, inicio):
    """Fechamento diário ajustado em formato longo: [data, symbol, close]."""
    d = yf.download(symbols, start=inicio, interval="1d", auto_adjust=True, progress=True)
    if isinstance(d.columns, pd.MultiIndex):
        close = d["Close"].copy()
    else:
        close = d[["Close"]].copy()
        close.columns = [symbols[0]]
    close = close.reset_index()
    col_data = close.columns[0]
    long = (close.melt(id_vars=col_data, var_name="symbol", value_name="close")
                 .dropna(subset=["close"])
                 .rename(columns={col_data: "data"}))
    long["data"] = pd.to_datetime(long["data"])
    return long


def _primeiro_ultimo_por_trimestre(long):
    """Primeiro e último fechamento (e datas) de cada (symbol, trimestre)."""
    long = long.sort_values("data").copy()
    long["trimestre"] = long["data"].dt.to_period("Q").astype(str)
    return long.groupby(["symbol", "trimestre"]).agg(
        preco_inicio=("close", "first"),
        preco_fim=("close", "last"),
        data_inicio=("data", "first"),
        data_fim=("data", "last"),
    ).reset_index()


def _carregar_tab(x):
    """Aceita DataFrame, caminho .csv/.parquet ou None — devolve DataFrame/None."""
    if x is None or isinstance(x, pd.DataFrame):
        return x
    if isinstance(x, str):
        df = pd.read_parquet(x) if x.endswith(".parquet") else pd.read_csv(x)
        df.columns = [str(col).lstrip("\ufeff") for col in df.columns]
        return df
    return x


def gerar_dados_backtest(caminho_carteira="melhores_por_trimestre.csv",
                         df_final=None, df_volumes=None, inicio="2020-01-01",
                         cache="precos_diarios_backtest.parquet"):
    # aceita DataFrame ou caminho (.csv/.parquet) em df_final e df_volumes
    df_final   = _carregar_tab(df_final)
    df_volumes = _carregar_tab(df_volumes)

    # 1) carteira (pivot Trimestre x Top_n) -> formato longo
    cart = pd.read_csv(caminho_carteira)
    col_tri = cart.columns[0]
    longc = (cart.melt(id_vars=col_tri, var_name="posicao", value_name="ticker")
                 .dropna(subset=["ticker"])
                 .rename(columns={col_tri: "trimestre"}))
    longc["posicao"] = longc["posicao"].str.replace("Top_", "", regex=False).astype(int)
    # trimestre de negociação = trimestre seguinte ao da seleção (sem look-ahead)
    longc["trimestre_holding"] = longc["trimestre"].apply(lambda q: str(pd.Period(q, "Q") + 1))

    tickers = sorted(longc["ticker"].dropna().unique())

    # 2) precos diarios ajustados (cacheados) + IBOV
    if os.path.exists(cache):
        precos_long = pd.read_parquet(cache)
        print(f"Precos diarios do cache ({cache}): {len(precos_long):,} linhas")
    else:
        precos_long = _close_diario_long([f"{t}.SA" for t in tickers], inicio)
        precos_long["symbol"] = precos_long["symbol"].str.replace(".SA", "", regex=False)
        precos_long.to_parquet(cache, index=False)
        print(f"Precos diarios salvos: {len(precos_long):,} linhas")
    ibov_long = _close_diario_long(["^BVSP"], inicio)

    # 3) primeiro/ultimo fechamento por trimestre
    acoes = _primeiro_ultimo_por_trimestre(precos_long).rename(columns={"symbol": "ticker"})
    ibov = (_primeiro_ultimo_por_trimestre(ibov_long)
            .assign(retorno_ibov=lambda d: d["preco_fim"] / d["preco_inicio"] - 1)
            [["trimestre", "retorno_ibov"]])

    # 4) monta o backtest (precos vem do trimestre de HOLDING)
    bt = longc.merge(
        acoes.rename(columns={"trimestre": "trimestre_holding"}),
        on=["ticker", "trimestre_holding"], how="left",
    )
    bt["retorno"] = bt["preco_fim"] / bt["preco_inicio"] - 1
    bt = bt.merge(ibov.rename(columns={"trimestre": "trimestre_holding"}),
                  on="trimestre_holding", how="left")

    # market_cap da CVM (ranking.csv), no trimestre da SELECAO
    if df_final is not None:
        mc = df_final.copy()
        mc["trimestre"] = pd.to_datetime(mc["DT_FIM_EXERC"]).dt.to_period("Q").astype(str)
        mc = mc[["ticker", "trimestre", "market_cap"]].drop_duplicates(["ticker", "trimestre"])
        bt = bt.merge(mc, on=["ticker", "trimestre"], how="left")

    # liquidez (R$/dia) no trimestre de holding
    if df_volumes is not None:
        vol = df_volumes.rename(columns={"trimestre": "trimestre_holding",
                                         "vol_medio_diario": "liquidez_rs"})
        bt = bt.merge(vol[["ticker", "trimestre_holding", "liquidez_rs"]],
                      on=["ticker", "trimestre_holding"], how="left")

    cols = ["trimestre", "trimestre_holding", "posicao", "ticker", "market_cap",
            "data_inicio", "preco_inicio", "data_fim", "preco_fim",
            "retorno", "retorno_ibov", "liquidez_rs"]
    bt = bt[[c for c in cols if c in bt.columns]].sort_values(["trimestre", "posicao"])
    return bt.reset_index(drop=True)


dados_backtest = gerar_dados_backtest("melhores_por_trimestre.csv",
                                      df_final="ranking.csv", df_volumes="volumes_trimestrais.parquet")
dados_backtest.to_csv("dados_backtest.csv", index=False, encoding="utf-8-sig")
print(f"Backtest salvo: {len(dados_backtest):,} linhas")
dados_backtest


Precos diarios do cache (precos_diarios_backtest.parquet): 104,073 linhas


[*********************100%***********************]  1 of 1 completed

Backtest salvo: 357 linhas


,trimestre,trimestre_holding,posicao,ticker,market_cap,data_inicio,preco_inicio,data_fim,preco_fim,retorno,retorno_ibov,liquidez_rs
0,2020Q1,2020Q2,1,ISAE4,8.931566e+08,2020-04-01,10.829729,2020-06-30,12.558632,0.159644,0.339439,2.376084e+07
1,2020Q1,2020Q2,2,SLCE3,1.257067e+09,2020-04-01,6.081589,2020-06-30,6.485562,0.066426,0.339439,2.859986e+07
2,2020Q1,2020Q2,3,WIZC3,1.008168e+09,2020-04-01,6.353097,2020-06-30,7.334750,0.154516,0.339439,6.312793e+06
3,2020Q1,2020Q2,4,TRIS3,8.070264e+08,2020-04-01,3.400102,2020-06-30,6.511379,0.915054,0.339439,1.415534e+07
4,2020Q1,2020Q2,5,BBSE3,3.181329e+10,2020-04-01,14.414836,2020-06-30,16.338011,0.133416,0.339439,7.910139e+07
...,...,...,...,...,...,...,...,...,...,...,...,...
352,2025Q4,2026Q1,11,ABEV3,2.340603e+11,2026-01-02,13.650000,2026-03-31,15.250000,0.117216,0.167704,4.149423e+08
353,2025Q4,2026Q1,12,PETR4,4.721737e+11,2026-01-02,29.794725,2026-03-31,47.219448,0.584826,0.167704,1.967042e+09
354,2025Q4,2026Q1,13,ITSA4,1.513125e+11,2026-01-02,11.488289,2026-03-31,13.953767,0.214608,0.167704,4.398086e+08
355,2025Q4,2026Q1,14,FIQE3,2.127132e+09,2026-01-02,4.890000,2026-03-31,7.010000,0.433538,0.167704,3.802858e+06


In [11]:
# Exemplos de seleção por trimestre usando o ranking composto
# (rank_composto = média simples dos ranks de ROIC, Graham, DL/EBITDA e CAGR lucro)

# Top 15 global do 4T/2023
top_global = selecionar_trimestre(df_final, "2023Q4", top_n=15, modo="global")

# Top 5 por setor
top_setor = selecionar_trimestre(df_final, "2023Q4", top_n=5, modo="por_setor")

# Melhor de cada setor
top_1_setor = selecionar_trimestre(df_final, "2023Q4", top_n=1, modo="1_por_setor")

top_global[["DENOM_CIA", "ticker", "SETOR_ATIV", "roic", "margem_seguranca",
            "dl_ebitda", "cagr_lucro", "rank_composto"]]


,DENOM_CIA,ticker,SETOR_ATIV,roic,margem_seguranca,dl_ebitda,cagr_lucro,rank_composto
0,ALLOS S.A.,ALOS3,Emp. Adm. Part. - Comércio (Atacado e Varejo),0.297001,0.666669,-0.349871,0.572962,11.00
1,KEPLER WEBER S.A.,KEPL3,Emp. Adm. Part. - Metalurgia e Siderurgia,0.426346,0.293125,-0.467674,0.969926,13.25
2,CAMBUCI S.A.,CAMB3,Têxtil e Vestuário,0.429489,0.396529,-0.084420,0.339800,16.00
3,TECHNOS S.A.,TECN3,Comércio (Atacado e Varejo),0.143263,0.683733,-0.440844,0.313657,17.75
4,SCHULZ S.A.,SHUL4,Metalurgia e Siderurgia,0.241401,0.342028,-0.415116,0.293992,17.75
5,BRADESPAR S.A.,BRAP4,Emp. Adm. Part. - Extração Mineral,0.241024,0.633213,-0.134623,0.097616,19.25
6,CIA HABITASUL DE PARTICIPACOES,HBTS5,Emp. Adm. Part. - Crédito Imobiliário,0.327700,0.862293,0.602283,NaN,20.00
7,PETROLEO BRASILEIRO S.A. PETROBRAS,PETR4,Petróleo e Gás,0.218853,0.635724,1.178895,0.362077,21.00
8,GRAZZIOTIN S.A.,CGRA4,Comércio (Atacado e Varejo),0.069869,0.754125,-1.475723,0.122114,21.50
9,METALURGICA RIOSULENSE S.A.,RSUL4,"Máquinas, Equipamentos, Veículos e Peças",0.449015,0.059244,-0.672402,NaN,21.75


In [ ]:
df_aalr3 = df_final[df_final["ticker"] == 'AALR3'].copy()

display(df_aalr3)

df_aalr3.to_csv("allr.csv")



,CNPJ_CIA,DENOM_CIA,ticker,SETOR_ATIV,is_financeiro,DT_FIM_EXERC,DT_INI_EXERC,meses_periodo,receita,receita_anualizada,...,margem_seguranca,cagr_lucro,cagr_receita,selic,flag_prejuizo,flag_pl_alto,flag_alavancagem,flag_cagr_neg,flag_margem_baixa,flag_roe_baixo
847,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2021-12-31,2021-01-01,12.0,1.136572e+09,1.136572e+09,...,-7.391735,-0.413991,0.036192,0.0925,False,True,False,True,True,True
1921,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2021-03-31,2021-01-01,3.0,2.851800e+08,1.140720e+09,...,0.114395,NaN,0.058292,0.0275,False,True,False,False,True,False
1944,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2021-09-30,2021-01-01,9.0,8.668690e+08,1.155825e+09,...,-0.570762,NaN,0.058292,0.0625,False,True,False,False,True,True
2332,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2021-06-30,2021-01-01,6.0,5.683920e+08,1.136784e+09,...,-0.030448,NaN,0.058292,0.0425,False,True,False,False,True,False
3705,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2022-03-31,2022-01-01,3.0,2.751570e+08,1.100628e+09,...,NaN,-0.413991,0.036192,0.1175,False,False,False,True,True,True
3732,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2022-06-30,2022-01-01,6.0,5.416740e+08,1.083348e+09,...,NaN,-0.413991,0.036192,0.1325,True,False,False,True,True,True
3757,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2022-09-30,2022-01-01,9.0,8.224290e+08,1.096572e+09,...,NaN,-0.413991,0.036192,0.1375,True,False,False,True,True,True
3881,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2025-06-30,2025-01-01,6.0,6.106300e+08,1.221260e+09,...,NaN,NaN,0.025530,0.1500,False,False,False,False,True,True
3902,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2023-06-30,2023-01-01,6.0,5.890660e+08,1.178132e+09,...,NaN,NaN,0.001320,0.1375,False,False,True,False,True,True
3925,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,AALR3,Serviços Médicos,False,2023-12-31,2023-01-01,12.0,1.179584e+09,1.179584e+09,...,NaN,NaN,0.018379,0.1175,True,False,False,False,True,True
